*Notebook 01 of 12 in `notebooks/scripts_text/`. Parameters: `analysis/scripts_text/config.yaml`; decisions and run notes: `analysis/scripts_text/LOG.md`. The static page `site/scripts/index.html` is built from the same outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/scripts_text -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.scripts_text import s01_groups as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 1 · Who counts as a success?

**What we do.** We define success as a film's gross divided by its budget, and use log(gross/budget) when we
need a number on a continuous scale: 0 means the film grossed its budget, 1 means about 2.7 times its budget.
From it we build three pairs of groups for the 1,095 clean 100k+ films that have a usable MovieSum script:

- **(A)** unprofitable (gross below budget) against everyone else. The most literal definition, but tiny.
- **(B)** the bottom quarter against the top quarter of log(gross/budget). The cut-offs are computed on all
  2,446 clean films, scripted or not, so a film's label doesn't depend on which other films happen to have a
  script.
- **(C)** the same quarters, but cut inside each primary-genre × decade cell. This compares a horror film from
  the 2000s with other horror films from the 2000s, not with dramas from the 1970s.

**Why.** Scattertext compares two groups, so we need to say which two. The course's "compared to what"
question starts here: a word can only be distinctive of success relative to a definition of failure.

**What it measures.** How much a film earned relative to what it cost to make, as reported in its Wikipedia
infobox.

**What it misses.** Gross minus budget is **not profit**: marketing isn't in the budget, cinemas keep a large
share of the gross (often around half), and nothing is adjusted for inflation (the ratio is unaffected by
inflation only because budget and gross are in the same year's dollars). The ratio also rewards very cheap
films: a $1 million horror film that grosses $50 million beats a $200 million blockbuster that grosses $1
billion.

**Primary analysis, fixed in advance:** definition (C), dialogue layer, character names removed, minimum
document frequency 5. Everything else in these notebooks is exploratory.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** Reproduction of the coverage numbers before anything else is computed.

In [ ]:
display(step.tables(res)["checks"])

,check,got,expected,match
0,clean_movies,2446,2446,True
1,clean_with_usable_script,1095,1095,True
2,gross_below_budget,67,67,True
3,global_quartile_bottom,243,243,True
4,global_quartile_top,306,306,True


**Table.** Group sizes under each definition. All films are in the 100k+ vote tier.

In [ ]:
display(step.tables(res)["sizes"])

,definition,meaning,tier,low (unsuccessful),high (successful),not in either group
0,A,unprofitable (gross < budget) vs all others,100k+,67,1028,0
1,B,"bottom vs top quartile of log(gross/budget), all films",100k+,243,306,546
2,C,bottom vs top quartile within genre × decade cells,100k+,237,292,566


**Figure.** log(gross/budget) for the 2,446 clean films, split by whether MovieSum has their script. Dashed lines: the quartile cuts used for (B) and break-even. The scripted films sit further right: the script sample over-represents hits, which is why (B) has 306 successful and only 243 unsuccessful scripts.

In [ ]:
step.figures(res)["coverage"].show()

**Figure.** Budget of the unsuccessful (orange) and successful (blue) scripts under each definition, log scale. Look for whether the two boxes overlap: where they don't, a word difference between the groups could be a budget difference.

In [ ]:
step.figures(res)["budget"].show()

**Figure.** Release year by group. A shift between the boxes means era could explain word differences (language changes over decades).

In [ ]:
step.figures(res)["year"].show()

**Figure.** Share of each primary genre in each group, with a fill pattern per genre so the chart reads without colour. Horror's share among successful films under (A) and (B) is the known micro-budget bias; (C) equalises genre by construction.

In [ ]:
step.figures(res)["genre_mix"].show()

**Table.** The budget-bias test: median budget, median year and horror share in each group, and a Mann-Whitney test of log budget between the two groups.

In [ ]:
display(step.tables(res)["bias"])

,definition,"median budget, low ($M)","median budget, high ($M)","median year, low","median year, high","horror share, low","horror share, high","Mann-Whitney p, log budget"
0,A,30.0,35.0,2009,2007,0.164,0.123,3.300000e-01
1,B,43.0,15.0,2008,2001,0.107,0.170,5.100000e-20
2,C,40.0,25.0,2008,2009,0.135,0.089,6.500000e-03


**Table.** Definition (C) within one genre or one decade alone. A stratum is used only if both groups have at least 30 scripts (the brief's rule).

In [ ]:
display(step.tables(res)["strata"])

,stratum,value,scripts,low,high,usable
0,genre,Action,84,14,20,no: a group has fewer than 30
1,genre,Adventure,203,40,57,yes
2,genre,Comedy,198,56,57,yes
3,genre,Drama,213,44,68,yes
4,genre,Horror,137,37,37,yes
5,genre,Other,4,2,0,no: a group has fewer than 30
6,genre,Sci-Fi,113,23,36,no: a group has fewer than 30
7,genre,Thriller,143,29,37,no: a group has fewer than 30
8,decade,1980s,78,16,26,no: a group has fewer than 30
9,decade,1990s,203,49,60,yes


**Table.** Genre × decade cells. A cell needs at least 20 clean films (scripted or not) to cut quartiles; scripts in smaller cells are left out of (C) and counted here.

In [ ]:
display(step.tables(res)["cells"])

,cell,scripts,cell_ok,low,high
9,Adventure | 2010s,87,True,19,25
21,Drama | 2010s,76,True,15,23
15,Comedy | 2010s,69,True,17,22
14,Comedy | 2000s,53,True,18,16
8,Adventure | 2000s,50,True,7,16
20,Drama | 2000s,49,True,11,10
41,Thriller | 2000s,49,True,9,13
27,Horror | 2010s,47,True,13,15
36,Sci-Fi | 2010s,45,True,10,15
19,Drama | 1990s,43,True,10,17


**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

All five coverage checks reproduce (clean_movies 2,446, clean_with_usable_script 1,095, gross_below_budget 67, global_quartile_bottom 243, global_quartile_top 306). Under (A) only 67 scripts are unsuccessful, so any word-level test on (A) has little power. Under (B) the median budget is $43.0M for the bottom quartile against $15.0M for the top (Mann-Whitney p = 5.1e-20), and horror makes up 17% of the top quartile against 11% of the bottom. Within genre × decade cells (C) the budget gap is $40.0M against $25.0M (p = 0.0065), so (C) removes the genre and era difference but not all of the budget difference; Step 4's matching handles budget. 32 of 47 genre × decade cells have at least 20 clean films; 76 scripts sit in cells too small to cut quartiles and are left out of (C). 7 single-genre or single-decade strata have fewer than 30 scripts in a group and are skipped: Action, Other, Sci-Fi, Thriller, 1980s, 2020s, pre-1980. The priority genre rule agrees with Àngela's first-listed rule for 49% of scripts.

**Why we keep three definitions for now.** (A) is the most literal reading of "unprofitable" but has so
few scripts that rare words can't be tested. (B) is larger and more robust, but mixes genres and eras, so it
partly measures "horror vs drama" or "1980s vs 2010s". (C) controls genre and era by construction, which is
why it's the primary analysis. None of them controls budget fully; Step 4 does that by matching.